<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Segment online-shop customers

**[C05-M09-L06 · Segment online-shop customers](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/)** · C05, Machine Learning: From Problem to Reliable Model · Module 9, case study 6 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** clean two years of real invoice lines from a UK online shop. You build recency, frequency and spend for each customer at a snapshot date, and group the customers with k-means. You choose the number of groups with inertia, silhouette and stability. Then you check the groups against what the customers did in the next six months. Then you compare the clusters with a simple quartile score.

| | |
|---|---|
| **Time** | About 120 minutes with the lesson page; the notebook runs in about one minute |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | k-means, scaling, inertia and silhouette, from [Unsupervised exploration](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/). Pipelines, from [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/). ROC AUC, from [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/). |
| **You do not need** | The course project. This notebook downloads its own data and uses the full dataset (no sample). |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L06-segment-online-shop-customers/segment-online-shop-customers-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It makes a folder `segment-online-shop-customers` with a `data` folder in it. It downloads `online_retail_ii.csv.gz` (15 MB) from the course's labs repository and checks the file's checksum (SHA-256). Everything this notebook writes goes into `segment-online-shop-customers`. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready:` and the file with its size. If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `segment-online-shop-customers` (in Colab: the **Files** panel on the left), then run the cell again.

The data is **Online Retail II** by Daqing Chen, from the UCI Machine Learning Repository ([doi.org/10.24432/C5CG6D](https://doi.org/10.24432/C5CG6D)). Its licence is [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course joined the two Excel sheets of the source into one compressed CSV file and changed nothing else; the [dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/case-studies/data/online_retail_ii.md) lists the details. It is real data of a UK online shop. Customer IDs are numbers only; do not try to identify anyone.

In [ ]:
# Setup: download the case-study data into segment-online-shop-customers/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/"
FILES = {
    "online_retail_ii.csv.gz": "c5ab45f73700a1411440c2cd6b7d85611d22cdc72b3594c2d29d37261e977e4f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("segment-online-shop-customers").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder segment-online-shop-customers and run again.")

os.chdir(WORK)  # work in the case-study folder
def size(path):
    n = path.stat().st_size
    return f"{n / 1e6:.1f} MB" if n >= 1e6 else f"{n / 1e3:.0f} KB"

print("Ready:", ", ".join(f"data/{n} ({size(Path('data', n))})" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Hannah, the CRM lead of a UK online shop, plans the second half of 2011. She has three tools. **VIP care** for about 750 customers is expensive. A **win-back voucher** for customers who have gone quiet costs margin. A **nurturing email** for everyone else is cheap. Owen, the finance manager, asks of every group: "If I spend money on this group, will they buy more than the others?" Hannah and Owen are made up; the data is real.

Your job: one row per customer with a segment name, and evidence that the segments mean something. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/#the-situation) tells the full story.

## 2. The data

Run the next cell. It loads the invoice lines. `Invoice` and `StockCode` are read as text, because some contain letters.

> **Check.** You should see `1,067,371 lines, 8 columns`, from 2009-12-01 to 2011-12-09.

In [ ]:
lines = pd.read_csv("data/online_retail_ii.csv.gz", dtype={"Invoice": str, "StockCode": str},
                    parse_dates=["InvoiceDate"])
print(f"{len(lines):,} lines, {lines.shape[1]} columns")
print("from", lines["InvoiceDate"].min(), "to", lines["InvoiceDate"].max())
lines.head(4)

**What this tells us.** One row is one product on one invoice. `Customer ID` shows as `13085.0`: a decimal number is a sign of missing values.

### First checks

Run the next cell. Each line counts one kind of problem.

> **Predict.** What share of the lines has no customer ID: under 5%, about 10% or more than 20%?

In [ ]:
missing_id = lines["Customer ID"].isna()
returns = lines["Invoice"].str.startswith("C")
print(f"no customer ID:        {missing_id.sum():>8,} lines ({missing_id.mean():.1%})")
print(f"exact repeated lines:  {lines.duplicated().sum():>8,}")
print(f"return lines (C...):   {returns.sum():>8,}")
print(f"quantity below 0:      {(lines['Quantity'] < 0).sum():>8,}")
print(f"price 0 or below:      {(lines['Price'] <= 0).sum():>8,}")
print(f"countries: {lines['Country'].nunique()}, United Kingdom {(lines['Country'] == 'United Kingdom').mean():.1%}")

> **Check.** You should see 243,007 lines with no customer ID (22.8%) and 34,335 exact repeated lines.

**What this tells us.** A quarter of the lines cannot be linked to a customer. There are more negative quantities (22,950) than return lines (19,494), so some negative quantities are not returns. Each problem needs a decision.

Run the next cell. It finds out when the repeats happen.

In [ ]:
repeated = lines[lines.duplicated()]
in_overlap = repeated["InvoiceDate"].between("2010-12-01", "2010-12-09 23:59")
print(f"{len(repeated):,} repeated lines; {in_overlap.sum():,} of them are from 1 to 9 December 2010")

> **Check.** You should see that 22,844 of the 34,335 repeats are from 1 to 9 December 2010.

**What this tells us.** The source file has one sheet per year, and both sheets contain those nine days. So most repeats are the same sales recorded twice by the way the file was made.

Run the next cell. It lists the product codes that do not start with five digits.

In [ ]:
odd = lines[~lines["StockCode"].str.match(r"^\d{5}")]
print(f"{len(odd):,} lines with a stock code that does not start with five digits")
(odd.groupby("StockCode")
    .agg(lines=("Invoice", "size"), example=("Description", "first"), price=("Price", "median"))
    .sort_values("lines", ascending=False).head(12))

**What this tells us.** Most of them are not goods: postage (`POST`, `DOT`), manual corrections (`M`), discounts (`D`), fees and gift vouchers. `DCGS0058` is a real product with an unusual code, so the cleaning lists the non-product codes by name.

Run the next cell. It shows the lines with more than 50,000 units.

In [ ]:
huge = lines["Quantity"].abs() > 50_000
lines.loc[huge, ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "InvoiceDate"]]

**What this tells us.** Customer 12346 ordered 74,215 jars and cancelled the order 16 minutes later. If you drop the returns and keep the order, this customer looks like a giant. So keep the returns, and compute **net** spend: purchases minus returns.

## 3. Clean the lines

The next cell applies four rules in order: exact repeats, no customer ID, non-product codes, and a price of 0 or below. It records how many lines each rule removes. Returns stay.

> **Check.** You should see 794,183 lines left, and 61 lines removed by the price rule.

In [ ]:
NOT_PRODUCTS = ["POST", "DOT", "C2", "C3", "M", "m", "D", "S", "B", "BANK CHARGES", "ADJUST",
                "ADJUST2", "AMAZONFEE", "CRUK", "TEST001", "TEST002", "GIFT"]
steps = []


def keep(df, rows_to_keep, reason):
    """Keep some lines, and record how many were removed and why."""
    steps.append({"removed because": reason, "lines removed": int((~rows_to_keep).sum()),
                  "lines left": int(rows_to_keep.sum())})
    return df[rows_to_keep]


clean = keep(lines, ~lines.duplicated(), "exact repeat of another line")
clean = keep(clean, clean["Customer ID"].notna(), "no customer ID")
product = ~clean["StockCode"].isin(NOT_PRODUCTS) & ~clean["StockCode"].str.startswith("gift_")
clean = keep(clean, product, "not a product (postage, fee, voucher)")
clean = keep(clean, clean["Price"] > 0, "price 0 or below")
pd.DataFrame(steps)

**What this tells us.** The order of the rules changes the counts. 6,207 lines had a price of 0 or below, but almost all of them had no customer ID. So the last rule removes only 61. Report the counts in the order you applied the rules.

Run the next cell. It adds the customer ID as text and the amount of each line, and checks the returns.

> **Check.** You should see 5,875 customers, and `negative quantity but not a return: 0`.

In [ ]:
clean = clean.assign(customer=clean["Customer ID"].astype(int).astype(str),
                     amount=clean["Quantity"] * clean["Price"])
returns = clean["Invoice"].str.startswith("C")
print(f"{clean['customer'].nunique():,} customers, {clean['Invoice'].nunique():,} invoices")
print(f"return lines: {returns.sum():,}; all with a negative quantity: {(clean.loc[returns, 'Quantity'] < 0).all()}")
print(f"negative quantity but not a return: {((clean['Quantity'] < 0) & ~returns).sum()}")
print("customer 12346, net amount per invoice (£):")
print(clean[clean["customer"] == "12346"].groupby("Invoice")["amount"].sum().round(2).to_string())

**What this tells us.** The mistaken order of customer 12346 and its return cancel out, so the customer's net spend is £169.36.

> **Your turn.** How many return lines (invoice starting with `C`) are in `clean`? Write the number into `return_lines`, computed from `clean`, then run the check.

In [ ]:
return_lines = int(clean["Invoice"].str.startswith("C").sum())
return_lines

In [ ]:
expect_hash('the number of return lines in clean', return_lines, '554fe5cca7a65543')

## 4. Frame the problem: the snapshot date

The unit is one customer. The decision date, the **snapshot**, is 1 June 2011: the segments use only the lines before it. The six months after it, June to November 2011, stay sealed until the forward check in section 9. Run the next cell.

> **Check.** You should see `history  541,106 lines, 4,932 customers` and `future   235,767 lines, 3,489 customers`.

In [ ]:
SNAPSHOT = pd.Timestamp("2011-06-01")
FUTURE_END = pd.Timestamp("2011-12-01")
history = clean[clean["InvoiceDate"] < SNAPSHOT]
future = clean[(clean["InvoiceDate"] >= SNAPSHOT) & (clean["InvoiceDate"] < FUTURE_END)]
for name, part in [("history", history), ("future", future)]:
    print(f"{name:8} {len(part):>7,} lines, {part['customer'].nunique():,} customers, "
          f"{part['InvoiceDate'].min():%Y-%m-%d} to {part['InvoiceDate'].max():%Y-%m-%d}")
print(f"not used: {(clean['InvoiceDate'] >= FUTURE_END).sum():,} lines from 1 to 9 December 2011")

**What this tells us.** The history is 18 months. Some future customers are new and cannot get a segment. Do not look at `future` again until section 9.

## 5. Recency, frequency and monetary value

For each customer: **recency** (days from the last purchase to the snapshot), **frequency** (the number of orders) and **monetary value** (net spend in £). Run the next cell.

> **Check.** You should see 4,908 customers with a purchase, 15 with a net spend of 0 or below, and `count 4893.0`.

In [ ]:
def rfm_table(history, snapshot):
    """One row per customer: recency, frequency and monetary value at the snapshot date."""
    purchases = history[~history["Invoice"].str.startswith("C")]
    table = purchases.groupby("customer").agg(last_purchase=("InvoiceDate", "max"),
                                              frequency=("Invoice", "nunique"))
    table["recency"] = (snapshot - table["last_purchase"]).dt.days
    table["monetary"] = history.groupby("customer")["amount"].sum().round(2)  # purchases minus returns
    return table[["recency", "frequency", "monetary"]]


rfm = rfm_table(history, SNAPSHOT)
print(f"{len(rfm):,} customers with a purchase; net spend 0 or below: {(rfm['monetary'] <= 0).sum()}")
rfm = rfm[rfm["monetary"] > 0]
rfm.describe().round(1)

**What this tells us.** The median customer spent £724, the mean £2,288 and the largest £383,393. The `.round(2)` stops tiny rounding errors (a net spend of 0.00000000000001) from passing the filter.

Run the next cell. It measures how unequal the customers are.

In [ ]:
print("skew:", rfm.skew().round(1).to_dict())
top_tenth = rfm["monetary"].nlargest(len(rfm) // 10).sum() / rfm["monetary"].sum()
print(f"the top 10% of customers by spend made {top_tenth:.1%} of the spend")
print(f"bought only once: {(rfm['frequency'] == 1).mean():.1%} of customers")

> **Check.** You should see a skew of 24.2 for `monetary`, and that the top 10% made 61.4% of the spend.

**What this tells us.** A small group really matters, which is good for Hannah. But k-means uses distances, and a few giants would decide them. Sections 7 and 11 deal with this.

## 6. A simple baseline: the quartile RFM score

Cut each number into four equal groups (quartiles), give 0 to 3 points, and add them up: 0 to 9 points. Recent is good, so recency counts the other way. Run the next cell.

> **Check.** You should see 376 customers with 0 points and 570 with 9 points.

In [ ]:
def quartile(values):
    """0 for the lowest quarter of the customers, 3 for the highest."""
    return pd.qcut(values.rank(method="first"), 4, labels=False)


rfm["rfm_score"] = quartile(-rfm["recency"]) + quartile(rfm["frequency"]) + quartile(rfm["monetary"])
rfm["rfm_score"].value_counts().sort_index().to_frame().T

**What this tells us.** Every score has 376 to 572 customers. This score is the baseline: the clusters must be at least as useful.

## 7. Prepare the numbers for k-means

The pipeline takes the log (`np.log1p`, log of 1 + x) and then scales each column. Run the next cell.

> **Check.** You should see the skew of `monetary` fall to 0.2, and every column with a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RFM = ["recency", "frequency", "monetary"]
prepare = make_pipeline(FunctionTransformer(np.log1p), StandardScaler())
X = prepare.fit_transform(rfm[RFM])
print("skew after the log:", np.log1p(rfm[RFM]).skew().round(1).to_dict())
pd.DataFrame(X, columns=RFM).describe().round(2).loc[["mean", "std", "min", "max"]]

**What this tells us.** The extremes are now about 5 standard deviations from the mean, not hundreds. No single customer decides the distances.

## 8. Choose the number of segments

The next cell tries k = 2 to 8. For each k it prints the **inertia**, the **silhouette** and the **stability**. Stability is the mean agreement (ARI) between 10 fits on different random 80% samples: 1 means the same groups, 0 means chance. It takes about 15 seconds.

> **Predict.** Which k will have the highest silhouette? Will that k be useful for Hannah's three tools?

In [ ]:
from sklearn.metrics import adjusted_rand_score, silhouette_score


def stability(X, k, runs=10):
    """Mean agreement (ARI) between k-means runs fitted on different random 80% samples."""
    rng = np.random.default_rng(0)
    labels = []
    for run in range(runs):
        rows = rng.choice(len(X), int(0.8 * len(X)), replace=False)
        labels.append(KMeans(n_clusters=k, n_init=10, random_state=run).fit(X[rows]).predict(X))
    return np.mean([adjusted_rand_score(a, b) for i, a in enumerate(labels) for b in labels[i + 1:]])


for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    print(f"k {k}: inertia {km.inertia_:4.0f}  silhouette {silhouette_score(X, km.labels_):.3f}  "
          f"stability {stability(X, k):.3f}  sizes {sorted(np.bincount(km.labels_).tolist(), reverse=True)}")

> **Check.** You should see `k 4: inertia 4209  silhouette 0.378  stability 0.964`.

The next cell draws inertia and silhouette.

In [ ]:
import matplotlib.pyplot as plt

ks = list(range(2, 9))
fits = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X) for k in ks]
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
left.plot(ks, [f.inertia_ for f in fits], marker="o")
left.set_xlabel("k"); left.set_ylabel("inertia")
right.plot(ks, [silhouette_score(X, f.labels_) for f in fits], marker="o", label="silhouette")
right.set_xlabel("k"); right.set_ylabel("silhouette"); right.legend()
plt.show()

**What this tells us.** Inertia bends at k = 4. Silhouette is highest at k = 2 (0.456), but two groups are only "active" and "not active". k = 4 has a local peak of silhouette and a high stability (0.964); k = 5 is less stable (0.866). All silhouette values are below 0.5, so the groups touch. You choose k = 4 because the evidence and the business agree.

### Fit the segments

Run the next cell. It fits the pipeline with k = 4 and shows the profile of each cluster.

> **Check.** You should see four clusters of 718, 1424, 761 and 1990 customers. The cluster numbers may be in another order on your computer.

In [ ]:
segmenter = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                          KMeans(n_clusters=4, n_init=10, random_state=0))
rfm["cluster"] = segmenter.fit_predict(rfm[RFM])
profile = rfm.groupby("cluster").agg(customers=("recency", "size"), recency=("recency", "median"),
                                     frequency=("frequency", "median"), monetary=("monetary", "median"),
                                     spend=("monetary", "sum"))
profile["share_of_spend"] = profile["spend"] / profile["spend"].sum()
profile.drop(columns="spend").round(3)

Hannah names the clusters from the profile:

- **Best customers**: 13 orders, recent. VIP care.
- **Recent occasional buyers**: recent, 3 orders. Nurture.
- **Quiet regulars**: 4 orders, but 174 days ago. Win-back offer.
- **Lapsed one-time buyers**: one order, long ago. A cheap reminder only.

The next cell gives each cluster its name from the profile, so the names are right on any computer.

In [ ]:
names = {profile["frequency"].idxmax(): "Best customers",
         profile["recency"].idxmax(): "Lapsed one-time buyers"}
rest = profile.drop(index=list(names))
names[rest["recency"].idxmin()] = "Recent occasional buyers"
names[rest["recency"].idxmax()] = "Quiet regulars"
rfm["segment"] = rfm["cluster"].map(names)
profile.index = profile.index.map(names)
profile.drop(columns="spend").round(3)

> **Check.** You should see `Best customers` with 761 customers and a share of spend of 0.658.

**What this tells us.** 761 customers (15.6%) made 65.8% of the spend. Run the next cell to see 500 customers by recency and spend.

In [ ]:
sample = rfm.sample(500, random_state=0)
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, group in sample.groupby("segment"):
    ax.scatter(group["recency"], group["monetary"], s=12, label=name)
ax.set_yscale("log")
ax.set_xlabel("recency (days since the last purchase)")
ax.set_ylabel("net spend, £ (log scale)")
ax.legend()
plt.show()

## 9. The forward check: the next six months

Now open the sealed future, once. For each customer: did they buy again in June to November 2011, and how much did they spend?

> **Predict.** What share of the best customers came back? And of the lapsed one-time buyers?

In [ ]:
future_spend = future.groupby("customer")["amount"].sum()
future_buyers = future.loc[~future["Invoice"].str.startswith("C"), "customer"].unique()
rfm["came_back"] = rfm.index.isin(future_buyers)
rfm["future_spend"] = future_spend.reindex(rfm.index).fillna(0)
check = rfm.groupby("segment").agg(customers=("came_back", "size"), came_back=("came_back", "mean"),
                                   future_spend=("future_spend", "mean"), total=("future_spend", "sum"))
check["share_of_future_spend"] = check["total"] / check["total"].sum()
print(f"came back overall: {rfm['came_back'].mean():.1%}; "
      f"new customers in the six months, not segmented: {len(set(future_buyers) - set(rfm.index)):,}")
check.drop(columns="total").sort_values("future_spend").round(3)

> **Check.** You should see that 92.6% of the best customers came back, against 27.2% of the lapsed one-time buyers.

In [ ]:
order = check.sort_values("future_spend").index
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(order, check.loc[order, "came_back"] * 100)
ax.set_ylabel("came back, June to November 2011 (%)")
ax.set_ylim(0, 100)
plt.show()

**What this tells us.** The segments hold: the best customers made 67.3% of the next six months' spend. But 57.7% of the quiet regulars came back without any offer, so a voucher for all of them gives margin away. 919 new customers have no segment yet.

> **Your turn.** How much did the lapsed one-time buyers spend in the six months, in total? Write it into `lapsed_future_spend`, rounded to whole pounds with `round()`, then run the check.

In [ ]:
lapsed_future_spend = round(rfm.loc[rfm["segment"] == "Lapsed one-time buyers", "future_spend"].sum())
lapsed_future_spend

In [ ]:
expect_hash('the total future spend of the lapsed one-time buyers', lapsed_future_spend if lapsed_future_spend is ... else int(lapsed_future_spend), '4bdcde3fee4ff4ec')

## 10. Clusters against the simple score

Run the next cell. It shows the come-back rate and the future spend for each RFM score.

In [ ]:
rfm.groupby("rfm_score").agg(customers=("came_back", "size"), came_back=("came_back", "mean"),
                             future_spend=("future_spend", "mean")).round(3)

> **Check.** You should see the come-back rate rise from 0.120 (score 0) to 0.956 (score 9).

The next cell compares the two methods: ROC AUC for "came back", and the VIP list of 761 customers chosen by each method. The segments are ranked by their past spend, not by the future.

In [ ]:
from sklearn.metrics import roc_auc_score

value_rank = rfm["segment"].map(profile["monetary"].rank())  # the segments in order of past spend
rankings = {"4 segments": value_rank, "RFM score 0-9": rfm["rfm_score"], "recency alone": -rfm["recency"]}
for name, ranking in rankings.items():
    print(f"{name:14} ROC AUC for 'came back': {roc_auc_score(rfm['came_back'], ranking):.3f}")

best = rfm[rfm["segment"] == "Best customers"]
top_by_score = rfm.sort_values(["rfm_score", "monetary"], ascending=False).head(len(best))
for name, group in [("Best customers", best), ("top by RFM score", top_by_score)]:
    print(f"{name:17} {len(group)} customers, came back {group['came_back'].mean():.1%}, "
          f"mean future spend £{group['future_spend'].mean():,.0f}, "
          f"{group['future_spend'].sum() / rfm['future_spend'].sum():.1%} of the future spend")
print("customers on both lists:", len(best.index.intersection(top_by_score.index)))

> **Check.** You should see ROC AUC 0.738 for the segments and 0.791 for the score, and 682 customers on both VIP lists.

**What this tells us.** The simple score is as good as the clusters, or better, for choosing the VIP list. The clustering found the groups and gave Hannah kinds of customer to name; the score is easier to check. Use both: the segments for the plan, the score for ranking. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/#clusters-against-the-simple-score) explains the choice.

## 11. Errors and slices

Run the next cell. It checks the come-back rates in the UK and in other countries, and counts two kinds of surprise.

In [ ]:
country = clean.groupby("customer")["Country"].agg(lambda c: c.mode()[0])
rfm["market"] = np.where(rfm.index.map(country) == "United Kingdom", "UK", "other")
slices = rfm.groupby(["segment", "market"]).agg(customers=("came_back", "size"),
                                                came_back=("came_back", "mean"))
print(slices.unstack().round(3))
best_gone = (rfm["segment"] == "Best customers") & ~rfm["came_back"]
lapsed_back = (rfm["segment"] == "Lapsed one-time buyers") & rfm["came_back"]
print(f"Best customers who did not come back: {best_gone.sum()}; lapsed buyers who came back: {lapsed_back.sum()}")

> **Check.** You should see 56 best customers who did not come back, and 542 lapsed buyers who came back.

**What this tells us.** The segments work the same in both markets. A segment describes a group, not a person: 56 best customers left anyway.

## 12. Failure case: no log

Owen leaves out the log step.

> **Predict.** What will the four clusters look like?

In [ ]:
no_log = make_pipeline(StandardScaler(), KMeans(n_clusters=4, n_init=10, random_state=0))
wrong = rfm[RFM].assign(cluster=no_log.fit_predict(rfm[RFM]))
wrong.groupby("cluster").agg(customers=("recency", "size"), recency=("recency", "median"),
                             frequency=("frequency", "median"), monetary=("monetary", "median"))

> **Check.** You should see two tiny clusters, of 32 and 3 customers.

**What this tells us.** Scaling does not remove a long tail. k-means builds clusters around a few giants, and the other 4,858 customers are cut only by recency. Take the log before you scale, and always check the cluster sizes.

## 13. Change one thing: five segments

> **Your turn.** Fit the same pipeline with `n_clusters=5`. Put the cluster sizes, sorted from largest to smallest, into a list `k5_sizes`. Then compare the profile with the four segments: which group is new?

In [ ]:
k5 = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                   KMeans(n_clusters=5, n_init=10, random_state=0))
rfm["k5"] = k5.fit_predict(rfm[RFM])
k5_sizes = sorted(np.bincount(rfm["k5"]).tolist(), reverse=True)
print(rfm.groupby("k5")[RFM].median())
print(pd.crosstab(rfm["segment"], rfm["k5"]))
k5_sizes

In [ ]:
expect_hash('the five cluster sizes', k5_sizes, '9b5749a92a89fbe4')

With k = 5, a middle group of good regulars (about 7 orders) splits off from the best customers and the quiet regulars. It is less stable (0.866) and gives Hannah no new action, so the experiment log rejects it.

## 14. The experiment log

Run the next cell. It writes `experiment_log.csv` with every attempt, including the ones that failed.

In [ ]:
from datetime import date

log = pd.DataFrame([
    ("E01", "Baseline: quartile RFM score 0-9", None, None, "ROC AUC 0.791; top 761: 93.3% came back", "baseline", "Simple, and a person can compute it."),
    ("E02", "k-means k=4, scaled, no log", None, None, "two clusters of 32 and 3 customers", "reject", "A few very large customers decide the clusters."),
    ("E03", "k-means, log + scale, k=2 to 8", "k=2 0.456, k=4 0.378", "k=4 0.964", None, "keep", "k=4 is stable and has four actions."),
    ("E04", "k-means k=5, log + scale", 0.327, 0.866, None, "reject", "A middle group from Best and Quiet; no new action."),
    ("E05", "Final: k=4, checked on Jun-Nov 2011", 0.378, 0.964, "Best 92.6% came back, Lapsed 27.2%", "keep", "The segments hold in the next six months."),
], columns=["id", "change", "silhouette", "stability", "result", "decision", "reason"])
log.insert(1, "date", date.today().isoformat())
log.to_csv("experiment_log.csv", index=False)
log[["id", "change", "decision"]]

## 15. Hand over the list

Run the next cell. It writes `segments.csv`: one row per customer, with the three numbers, the score and the segment, for Hannah's email tool.

> **Check.** You should see 761 best customers, and customer 12346 as a lapsed one-time buyer with a frequency of 3.

In [ ]:
deliverable = rfm[RFM + ["rfm_score", "segment"]].reset_index()
deliverable.to_csv("segments.csv", index=False)
print(deliverable["segment"].value_counts().to_string())
deliverable.head(3)

**What this tells us.** Customer 12346 has a frequency of 3, but one of the orders was the mistaken order of 74,215 jars. A cancelled order still counts as an order. That is one of the limits on the lesson page.

## 16. Recommendations, limits and your portfolio

The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/#recommendations) has the recommendations for Hannah and Owen and the limits. It also has a guide to turn this notebook into a portfolio project: a README, this notebook, `experiment_log.csv`, `segments.csv` and a short model card. The two files are in the folder `segment-online-shop-customers`.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/#knowledge-check).

In this case study you cleaned 1,067,371 invoice lines, with a reason for every rule. You built RFM for 4,893 customers at a snapshot date, and chose k = 4 with inertia, silhouette and stability. The forward check showed that 92.6% of the best customers came back, against 27.2% of the lapsed buyers. A simple quartile score did as well for the VIP list. Next: wholesale buyer types, where the best silhouette points to a useless answer.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Find wholesale buyer types](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real (UCI Online Retail II, CC BY 4.0); the people in the story are made up.